# Experiment 11 — Paillier Partially Homomorphic Encryption
Additive homomorphism: Dec(Enc(m₁) × Enc(m₂) mod n²) = m₁ + m₂

## Helpers — Primes, LCM, Inverse

In [ ]:
import random, math, secrets

def ext_gcd(a,b):
    if a==0: return b,0,1
    g,x,y=ext_gcd(b%a,a); return g,y-(b//a)*x,x
def mod_inv(a,m): _,x,_=ext_gcd(a%m,m); return x%m
def lcm(a,b): return a*b//math.gcd(a,b)

def miller_rabin(n,k=10):
    if n<2: return False
    if n in(2,3): return True
    if n%2==0: return False
    r,d=0,n-1
    while d%2==0: r+=1; d//=2
    for _ in range(k):
        a=random.randrange(2,n-1); x=pow(a,d,n)
        if x in(1,n-1): continue
        for _ in range(r-1):
            x=pow(x,2,n)
            if x==n-1: break
        else: return False
    return True

def gen_prime(bits):
    while True:
        n=secrets.randbits(bits); n|=(1<<(bits-1))|1
        if miller_rabin(n): return n

def L(u,n): return (u-1)//n

print("Helpers defined ✓")


## Paillier Key Generation

In [ ]:
def paillier_keygen(bits=256):
    while True:
        p=gen_prime(bits); q=gen_prime(bits)
        if p==q: continue
        n=p*q; n2=n*n; lam=lcm(p-1,q-1)
        g=n+1; mu=mod_inv(L(pow(g,lam,n2),n),n)
        return (n,g),(lam,mu),p,q

print("Generating Paillier keys (256-bit primes)...")
pub,priv,p,q=paillier_keygen(256)
n,g=pub; lam,mu=priv
print(f"p = {p}")
print(f"q = {q}")
print(f"n = {n}")
print(f"g = n+1")
print(f"λ = {lam}")
print(f"μ = {mu}")


## Encrypt / Decrypt

In [ ]:
def paillier_enc(m,pub):
    n,g=pub; n2=n*n
    while True:
        r=secrets.randbelow(n)
        if r>0 and math.gcd(r,n)==1: break
    return (pow(g,m,n2)*pow(r,n,n2))%n2

def paillier_dec(c,pub,priv):
    n,_=pub; lam,mu=priv
    return (L(pow(c,lam,n*n),n)*mu)%n

m=42; ct=paillier_enc(m,pub); dt=paillier_dec(ct,pub,priv)
print(f"Plaintext  : {m}")
print(f"Ciphertext : {ct}")
print(f"Decrypted  : {dt}")
assert dt==m; print("✓ Basic roundtrip verified")


## Homomorphic Addition

In [ ]:
def homo_add(c1,c2,n): return (c1*c2)%(n*n)
def homo_add_const(c,k,pub):
    n,g=pub; return (c*pow(g,k,n*n))%(n*n)
def homo_mul_const(c,k,n): return pow(c,k,n*n)

m1,m2=25,17
c1=paillier_enc(m1,pub); c2=paillier_enc(m2,pub)
c_sum=homo_add(c1,c2,n)
dec_sum=paillier_dec(c_sum,pub,priv)
print(f"m1={m1}, m2={m2}, m1+m2={m1+m2}")
print(f"Dec(Enc(m1)×Enc(m2)) = {dec_sum}")
assert dec_sum==m1+m2; print("✓ Homomorphic addition verified")

k=100; c_c=homo_add_const(c1,k,pub)
dec_c=paillier_dec(c_c,pub,priv)
print(f"Dec(Enc({m1}) × g^{k}) = {dec_c}  (expected {m1+k})")
assert dec_c==m1+k; print("✓ Constant addition verified")

k2=7; c_m=homo_mul_const(c1,k2,n)
dec_m=paillier_dec(c_m,pub,priv)
print(f"Dec(Enc({m1})^{k2}) = {dec_m}  (expected {m1*k2})")
assert dec_m==m1*k2; print("✓ Scalar multiplication verified")


## Private Vote Tally (Application)

In [ ]:
votes=[1,0,1,1,0,1,0,1]
enc_votes=[paillier_enc(v,pub) for v in votes]
tally_enc=enc_votes[0]
for ev in enc_votes[1:]: tally_enc=homo_add(tally_enc,ev,n)
tally=paillier_dec(tally_enc,pub,priv)
print(f"Votes            : {votes}")
print(f"Expected yes sum : {sum(votes)}")
print(f"Homomorphic tally: {tally}")
assert tally==sum(votes)
print("✓ Private tally — individual votes never decrypted!")
print()
print("Additive HE ONLY (not multiplicative → 'partially' homomorphic)")
print("IND-CPA secure under Decisional Composite Residuosity (DCR) assumption")
